In [1]:
!pip install -q keras-hub

In [2]:
!wget "https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/21154/1243559/compressed/tfrecords-jpeg-224x224.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1784523107&Signature=XjHKY%2FJiZb4TZKmJ4Sp%2FXmFw37qyu0srE71e3ZHt5Qg%2BlBrineNmpmeBR2Z%2FTYFY2VpXOQ4PRw%2FPicIo62Ox0XMAburagYL1pQ01SiBTzAzkRFDVF%2F8oCqrG9o3DC1C6D3%2Bzqn6ZKUU5DnKdKw0dpIEQhl%2BMn%2B%2Bm82Wn%2F85LReUgOEbjCMQthzZIyikKWLcNhTDytn4c4YybsB6DpxoKoYBvQBmGLnKRuDLA%2F2mW3nrCz0ZypnuY6sgUxdxrlmlz97LGdWb0gmF7KX78RotX8Ya1cIDntdDNzY7%2Bm8KZWhyN43Z53fdrrPB%2FLjqZ9e%2B9MBK90eqcGm0ifoYMUCbEvQ%3D%3D&response-content-disposition=attachment%3B+filename%3Dtfrecords-jpeg-224x224.zip" -o "tfrecords-jpeg-224x224.zip"

In [4]:
import shutil

# Extract the entire ZIP file into a specific directory
shutil.unpack_archive("tfrecords-jpeg-224x224.zip", "tfrecords-jpeg-224x224")

In [5]:
import tensorflow as tf

In [6]:
print("TF version:", tf.__version__)
strategy = tf.distribute.get_strategy()
print("GPU:", tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
import os

In [8]:
IMAGE_SIZE = [224,224]
NUM_CLASSES = 104
BATCH_SIZE = 16

folder_path = '/content/tfrecords-jpeg-224x224'

TRAIN_FILES = [
    os.path.join(folder_path, "train", f)
    for f in os.listdir(os.path.join(folder_path, "train"))
    if f.endswith(".tfrec")
]

VAL_FILES = [
    os.path.join(folder_path, "val", f)
    for f in os.listdir(os.path.join(folder_path, "val"))
    if f.endswith(".tfrec")
]

print("files:", len(TRAIN_FILES), len(VAL_FILES))

files: 16 16


In [9]:
import re
import numpy as np

# Decode a JPEG string into a normalized float image
def decode_image(image_data):
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

# Parse a labeled example (image + class)
def read_labeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "class": tf.io.FixedLenFeature([], tf.int64)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), tf.cast(ex["class"], tf.int32)

# Parse an unlabeled example (image + id)
def read_unlabeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "id": tf.io.FixedLenFeature([], tf.string)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), ex["id"]

# Count images from the number encoded in each filename
def count_items(filenames):
    n = [int(re.compile(r"-([0-9]*)\.").search(f).group(1)) for f in filenames]
    return int(np.sum(n))

In [10]:
# Build a TFRecord dataset, optionally deterministic
def load_dataset(filenames, labeled=True, ordered=False):
    ds = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTOTUNE)
    if ordered:
        opt = tf.data.Options(); opt.deterministic = True
        ds = ds.with_options(opt)
    ds = ds.map(read_labeled if labeled else read_unlabeled, num_parallel_calls=AUTOTUNE)
    return ds

# Training set: shuffled, repeated, batched
def get_training_dataset():
    ds = load_dataset(TRAIN_FILES, labeled=True)
    return ds.repeat().shuffle(2048).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Validation set: ordered, batched
def get_validation_dataset():
    return load_dataset(VAL_FILES, labeled=True, ordered=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)

NUM_TRAIN, NUM_VAL= count_items(TRAIN_FILES), count_items(VAL_FILES)
STEPS_PER_EPOCH = NUM_TRAIN // BATCH_SIZE
print("train/val:", NUM_TRAIN, NUM_VAL, "| steps/epoch:", STEPS_PER_EPOCH)

train/val: 12753 3712 | steps/epoch: 797


In [11]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomBrightness(0.20),
], name="augmentation")

In [13]:
# keras_hub.__version__

In [14]:
import keras_hub

# Load directly from the official Keras repository on Hugging Face
vit = keras_hub.models.ImageClassifier.from_preset(
    "hf://keras/vit_base_patch16_224_imagenet",
    load_weights=True
)


backbone = vit.backbone
backbone.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.layers.Rescaling(1./255)(x)

x = backbone(x)

x = tf.keras.layers.Lambda(lambda v: v[:, 0, :])(x)

x = tf.keras.layers.Dropout(0.4)(x)

x = tf.keras.layers.Dense(
    512,
    activation="gelu"
)(x)

x = tf.keras.layers.Dropout(0.4)(x)

x = tf.keras.layers.Dense(
    256,
    activation="gelu"
)(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

config.json:   0%|          | 0.00/909 [00:00<?, ?B/s]

task.json:   0%|          | 0.00/3.82k [00:00<?, ?B/s]

task.weights.h5: reconstructing file:   0%|          |  0.00B /  347MB            

task.weights.h5: downloading bytes:           |  0.00B            

model.weights.h5: reconstructing file:   0%|          |  0.00B /  344MB            

model.weights.h5: downloading bytes:           |  0.00B            

In [15]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vi_t_backbone (ViTBackbone)     │ (None, 197, 768)       │    85,798,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 104)            │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,350,440 (329.40 MB)

 Trainable params: 551,784 (2.10 MB)

 Non-trainable params: 85,798,656 (327.30 MB)

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_acc"
        )
    ]
)

In [17]:
def lrfn(epoch):
  start, mx, mn, warm = 1e-5, 2e-4, 1e-6, 3
  if epoch < warm:
      return start + (mx - start) * epoch / warm
  return mn + (mx - mn) * (0.8 ** (epoch - warm))

In [18]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnet.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=1)
]

In [19]:
history = model.fit(
    get_training_dataset(),
    validation_data=get_validation_dataset(),
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=NUM_VAL // BATCH_SIZE,
    epochs=15,
    callbacks=callbacks
)


Epoch 1: LearningRateScheduler setting learning rate to 1e-05.
Epoch 1/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 346s 411ms/step - accuracy: 0.0294 - loss: 4.8724 - top5_acc: 0.1044 - val_accuracy: 0.2734 - val_loss: 4.0138 - val_top5_acc: 0.4297 - learning_rate: 1.0000e-05

Epoch 2: LearningRateScheduler setting learning rate to 7.333333333333333e-05.
Epoch 2/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 329s 413ms/step - accuracy: 0.3166 - loss: 3.2176 - top5_acc: 0.5034 - val_accuracy: 0.7592 - val_loss: 1.4672 - val_top5_acc: 0.8887 - learning_rate: 7.3333e-05

Epoch 3: LearningRateScheduler setting learning rate to 0.00013666666666666666.
Epoch 3/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 329s 412ms/step - accuracy: 0.6276 - loss: 1.6711 - top5_acc: 0.8029 - val_accuracy: 0.9046 - val_loss: 0.4697 - val_top5_acc: 0.9666 - learning_rate: 1.3667e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.0002.
Epoch 4/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 332s 417ms/step - accuracy: 0.7560 - loss: 1.0251 - top5_acc: 0.